# Intersection / Overlap classifier — V2 MAX RECALL

Notebook mejorado para entrenar un **modelo independiente de victim–perpetrator overlap** (`INTERSECT`) evitando leakage:

- Se crea `INTERSECT = (VÍCTIMA > 0) & (PERPETRADOR > 0)`.
- Se hace **train/test split antes de escalar y antes de PCA**.
- `MinMaxScaler`, medias de centrado y `PCA` se ajustan **solo con train**.
- Test se transforma con el scaler/PCA ya ajustado en train.
- Se usa el mismo criterio PCA de los otros modelos: **componentes hasta preservar ≥95% de varianza**.
- Se guardan predicciones, métricas, binarios TP/FP/TN/FN, varianza PCA y loadings.

## Mejoras introducidas en esta versión (V2 MAX RECALL)

`INTERSECT` es la clase más rara del proyecto (solo adolescentes que son víctima Y perpetrador a la vez), lo que hace que minimizar los falsos negativos sea especialmente crítico y difícil. Esta versión aplica las mismas mejoras de la versión V3 del clasificador de perpetrador, adaptadas al pipeline train-only PCA de este notebook:

**1. Focal Loss (nuevo en V2):** Sustituye la binary cross-entropy estándar por focal loss (`gamma=2`, `alpha=0.75`). Al ser `INTERSECT` la clase con menor soporte, la focal loss concentra el gradiente en los casos de intersección que el modelo tiende a perder, siendo la mejora de mayor impacto teórico para este objetivo.

**2. SMOTE en entrenamiento (nuevo en V2):** Se aplica SMOTE exclusivamente sobre el conjunto de entrenamiento (nunca sobre test, para no contaminar la evaluación). Genera muestras sintéticas interpoladas de casos de intersección, complementando el efecto del `class_weight`. Con la clase tan minoritaria, SMOTE es especialmente valioso aquí.

**3. Ensemble multi-semilla (nuevo en V2):** Cada configuración del grid se entrena con `n_seeds` semillas distintas y las probabilidades se promedian. Con muestras positivas pequeñas, la varianza entre ejecuciones es alta; el ensemble la reduce y produce probabilidades más estables para el barrido de umbrales.

**4. `screening_score` compuesto (nuevo en V2):** Reemplaza el criterio `score_tuple` (gate binario `sensitivity >= 0.80`) por un score continuo con penalizaciones sobre especificidad y precisión mínimas. El gate binario es frágil: si ningún modelo alcanza 0.80, todos se ordenan solo por `bal_acc`. El score continuo siempre distingue bien entre candidatos.

**5. `ReduceLROnPlateau` (nuevo en V2):** Reduce el learning rate automáticamente cuando `val_recall` se estanca, permitiendo al optimizador encontrar mejores mínimos sin más épocas.

**6. Semillas fijas completas (nuevo en V2):** Se fijan `PYTHONHASHSEED`, `numpy`, `random` y `tf.random` tanto globalmente como dentro de cada run del ensemble para garantizar reproducibilidad entre ejecuciones.

**7. Umbral extendido hasta 0.10 (nuevo en V2):** Con probabilidades calibradas por el ensemble, umbrales muy bajos son más fiables y pueden recuperar casos de intersección adicionales, siempre que la especificidad mínima se mantenga.

**8. `recall_priority_weight` (nuevo en V2):** Multiplica el `class_weight` natural por un factor adicional para reforzar el foco en no perder casos de intersección, complementando el efecto de SMOTE.

---

**Heredado de la versión FIXED:**
- Train-only PCA sin leakage.
- Split antes de scaler/PCA.
- Guardado del mejor modelo real, no del último entrenado.
- `threshold_used` en el CSV de predicciones.

El resultado debe interpretarse como un **modelo de screening muy sensible**, útil para detectar posibles casos de overlap y priorizar revisión posterior, no como una clasificación diagnóstica definitiva.

In [ ]:
#%pip install torch
#%pip install imbalanced-learn
import torch
torch.cuda.is_available()

In [ ]:
import os
import json
import random
import itertools
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import joblib
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ============================================================
# Configuración general
# ============================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("Determinismo TensorFlow activado.")
except Exception as exc:
    print("No se pudo activar determinismo estricto:", exc)

OUTPUT_DIR = Path("./content/intersection_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PCA_VARIANCE_THRESHOLD = 0.95
TEST_SIZE = 0.25
BATCH_SIZE = 128

# Si quieres una búsqueda rápida para probar, pon FAST_GRID=True.
FAST_GRID = True

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("Output dir:", OUTPUT_DIR)


# ============================================================
# Focal Loss
# ============================================================
def focal_loss(gamma=2.0, alpha=0.60):
    """
    Focal loss para clasificación binaria.
    gamma: factor de modulación (penaliza más los ejemplos difíciles).
    alpha: peso de la clase positiva. 0.60 es más conservador que 0.75 cuando
           ya se usa SMOTE, evitando que se acumulen demasiados sesgos positivos.
    """
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
        focal_weight = alpha_t * tf.pow(1.0 - p_t, gamma)
        bce = -tf.math.log(p_t)
        return tf.reduce_mean(focal_weight * bce)
    return loss_fn


# ============================================================
# Helpers
# ============================================================
def report_to_dataframe(y_true, y_pred):
    report_dict = classification_report(
        y_true, y_pred, digits=3, output_dict=True, zero_division=0
    )
    rows = []
    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):
            rows.append({
                "label": label,
                "precision": metrics.get("precision"),
                "recall": metrics.get("recall"),
                "f1-score": metrics.get("f1-score"),
                "support": metrics.get("support"),
            })
        else:
            rows.append({
                "label": label,
                "precision": None,
                "recall": None,
                "f1-score": metrics,
                "support": report_dict["weighted avg"]["support"],
            })
    return pd.DataFrame(rows)


def binary_metrics_dataframe(y_true, y_pred, label_name="intersection"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else np.nan
    acc = (tp + tn) / (tp + fp + tn + fn)
    bal_acc = (sensitivity + specificity) / 2 if (sensitivity is not np.nan and specificity is not np.nan) else np.nan
    return pd.DataFrame([{
        "outcome": label_name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision_ppv": ppv,
        "recall_sensitivity": sensitivity,
        "specificity": specificity,
        "npv": npv,
        "f1_positive": f1,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "support": int(tp + fp + tn + fn),
    }])


def fit_train_only_pca(X_train_df, X_test_df, threshold=0.95, output_dir=OUTPUT_DIR):
    """
    Fit MinMaxScaler + PCA SOLO con train y transforma train/test.
    Replica la lógica de los notebooks originales: MinMaxScaler -> centrar con medias -> PCA.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_df.values)
    X_test_scaled = scaler.transform(X_test_df.values)

    means_array = X_train_scaled.mean(axis=0)
    means = pd.Series(means_array, index=X_train_df.columns, name="mean")

    X_train_centered = X_train_scaled - means_array
    X_test_centered = X_test_scaled - means_array

    pca = PCA(n_components=X_train_df.shape[1], random_state=SEED)
    X_train_pca_all = pca.fit_transform(X_train_centered)
    X_test_pca_all = pca.transform(X_test_centered)

    cols_all = [f"PC{i+1}" for i in range(X_train_df.shape[1])]
    var_ratio = pca.explained_variance_ratio_
    cum_var = var_ratio.cumsum()
    df_variance = pd.DataFrame({
        "PC": cols_all,
        "explained variance": var_ratio,
        "cumulative variance": cum_var,
    })

    n_components = int(np.searchsorted(cum_var, threshold) + 1)
    cols = cols_all[:n_components]

    X_train_pca = pd.DataFrame(X_train_pca_all[:, :n_components], index=X_train_df.index, columns=cols)
    X_test_pca = pd.DataFrame(X_test_pca_all[:, :n_components], index=X_test_df.index, columns=cols)

    loadings = pd.DataFrame(
        pca.components_,
        columns=X_train_df.columns,
        index=cols_all,
    )

    top_rows = []
    for pc in cols:
        s = loadings.loc[pc].sort_values(key=np.abs, ascending=False).head(10)
        row = {"PC": pc}
        for i, (var, val) in enumerate(s.items(), start=1):
            row[f"var_{i}"] = var
            row[f"loading_{i}"] = val
        top_rows.append(row)
    top_loadings = pd.DataFrame(top_rows)

    joblib.dump(scaler, output_dir / "scaler_minmax_train_only.pkl")
    joblib.dump(pca, output_dir / "pca_train_only.pkl")
    means.to_csv(output_dir / "pca_train_means.csv", encoding="utf-8-sig")
    df_variance.to_csv(output_dir / "df_PCA_variance_intersection.csv", index=False, encoding="utf-8-sig")
    loadings.to_csv(output_dir / "pca_loadings_intersection.csv", encoding="utf-8-sig")
    top_loadings.to_csv(output_dir / "pca_top_loadings_intersection.csv", index=False, encoding="utf-8-sig")
    X_train_pca.to_csv(output_dir / "X_train_pca.csv", index=True, encoding="utf-8-sig")
    X_test_pca.to_csv(output_dir / "X_test_pca.csv", index=True, encoding="utf-8-sig")

    print(f"PCA threshold: {threshold}")
    print(f"PCA components retained: {n_components}")
    print(f"Cumulative variance retained: {cum_var[n_components-1]:.4f}")

    return X_train_pca, X_test_pca, df_variance, loadings, top_loadings, scaler, pca, means


def train_overlap_nn(
    X_train_df,
    X_test_df,
    y_train_s,
    y_test_s,
    output_dir=OUTPUT_DIR,
    batch_size=128,
    fast_grid=False,
    seed=42,
    recall_priority_weight=1.0,   # 1.0 cuando se usa SMOTE: evita acumular sesgos positivos
    min_specificity=0.40,          # floor más alto que la v1 para evitar soluciones degeneradas
    min_precision=0.25,
    target_recall=0.80,
    use_smote=True,
    use_focal_loss=True,
    focal_gamma=2.0,
    focal_alpha=0.60,              # más conservador que 0.75 cuando ya hay SMOTE
    n_seeds=3
):
    """
    Entrena red neuronal para INTERSECT sobre PCA ya ajustado solo con train.

    Corrección anti-degeneración (v2.1):
    - min_specificity subido a 0.40: modelos con especificidad < 0.40 quedan excluidos con floor duro.
    - focal_alpha bajado a 0.60: SMOTE ya balancea; acumular focal_alpha=0.75 + recall_priority_weight
      era excesivo y llevaba al modelo a clasificar casi todo como positivo.
    - recall_priority_weight=1.0 por defecto cuando use_smote=True.
    - screening_score con FLOOR DURO: si specificity < min_specificity o precision < min_precision,
      el score se fuerza a -inf y el modelo queda descartado, sin importar su recall.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Estrategia TPU/GPU/CPU
    using_tpu = False
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        using_tpu = True
        print("👾 TPU inicializado:", resolver.master())
    except Exception:
        strategy = tf.distribute.get_strategy()
        print("⚠️ No se encontró TPU, usando", type(strategy).__name__)

    try:
        policy_name = "mixed_bfloat16" if using_tpu else "float32"
        mixed_precision.set_global_policy(policy_name)
        print("Política de precisión:", mixed_precision.global_policy())
    except Exception as e:
        print("No se pudo ajustar mixed precision:", e)

    X_train_raw = X_train_df.values.astype("float32")
    X_test = X_test_df.values.astype("float32")
    y_train_raw = y_train_s.astype(int).values
    y_test = y_test_s.astype(int).values

    print(f"Train n={len(y_train_raw)} | Test n={len(y_test)}")
    print("Distribución train:", dict(zip(*np.unique(y_train_raw, return_counts=True))))
    print("Distribución test:", dict(zip(*np.unique(y_test, return_counts=True))))

    # SMOTE sobre entrenamiento (nunca sobre test)
    if use_smote:
        try:
            from imblearn.over_sampling import SMOTE
            sm = SMOTE(random_state=seed, k_neighbors=min(5, Counter(y_train_raw)[1] - 1))
            X_train, y_train = sm.fit_resample(X_train_raw, y_train_raw)
            X_train = X_train.astype("float32")
            print(f"SMOTE aplicado. Train balanceado: {dict(zip(*np.unique(y_train, return_counts=True)))}")
        except ImportError:
            print("imbalanced-learn no disponible. Instala con: pip install imbalanced-learn")
            print("Continuando sin SMOTE.")
            X_train, y_train = X_train_raw, y_train_raw
    else:
        X_train, y_train = X_train_raw, y_train_raw

    # class_weight — con SMOTE el ratio ya es ~1:1, recall_priority_weight=1.0 no añade sesgo extra
    unique_classes, counts = np.unique(y_train, return_counts=True)
    class_weight = {int(cls): 1.0 for cls in unique_classes}
    if set(unique_classes) == {0, 1}:
        n0 = counts[unique_classes == 0].sum()
        n1 = counts[unique_classes == 1].sum()
        class_weight = {0: 1.0, 1: float((n0 / n1) * recall_priority_weight)}
    print("Class weight:", class_weight)

    # Dataset de test (fijo, no se re-balancea)
    test_ds = (
        tf.data.Dataset.from_tensor_slices((X_test, y_test))
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    # Callbacks
    def make_callbacks():
        return [
            EarlyStopping(
                monitor="val_recall", mode="max",
                patience=12, restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="val_recall", mode="max",
                factor=0.5, patience=6,
                min_lr=1e-5, verbose=0
            )
        ]

    # Grid
    if fast_grid:
        grid_params = {
            "u1": [64, 32], "a1": ["relu", "tanh"], "d1": [0.3, 0.2],
            "u2": [16, 8],  "a2": ["relu", "tanh"], "d2": [0.1, 0.0],
        }
    else:
        grid_params = {
            "u1": [64, 32, 16], "a1": ["relu", "linear", "tanh", "sigmoid"], "d1": [0.3, 0.2, 0.1],
            "u2": [16, 8, 4],   "a2": ["relu", "linear", "tanh", "sigmoid"], "d2": [0.1, 0.05, 0.0],
        }

    # Umbrales extendidos hasta 0.10
    thresholds = np.round(np.arange(0.10, 0.56, 0.05), 2).tolist()

    # Función de pérdida
    loss_fn = focal_loss(gamma=focal_gamma, alpha=focal_alpha) if use_focal_loss else "binary_crossentropy"
    print(f"Loss: {'focal_loss(gamma={}, alpha={})'.format(focal_gamma, focal_alpha) if use_focal_loss else 'binary_crossentropy'}")
    print(f"min_specificity (floor duro): {min_specificity} | min_precision (floor duro): {min_precision}")

    def build_model(u1, a1, d1, u2, a2, d2, run_seed):
        tf.random.set_seed(run_seed)
        m = models.Sequential([
            layers.Input(shape=(X_train.shape[1],)),
            layers.Dense(u1, activation=a1),
            layers.Dropout(d1, seed=run_seed),
            layers.Dense(u2, activation=a2),
            layers.Dropout(d2, seed=run_seed),
            layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss=loss_fn,
            metrics=[
                tf.keras.metrics.Recall(name="recall"),
                tf.keras.metrics.Precision(name="precision"),
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            ],
        )
        return m

    ensemble_seeds = [seed + i * 100 for i in range(n_seeds)]
    print(f"Ensemble multi-semilla: {ensemble_seeds}")

    results = []
    best = None

    with strategy.scope():
        for u1, a1, d1, u2, a2, d2 in itertools.product(
            grid_params["u1"], grid_params["a1"], grid_params["d1"],
            grid_params["u2"], grid_params["a2"], grid_params["d2"]
        ):
            # Ensemble: entrenar n_seeds veces y promediar probabilidades
            prob_accumulator = np.zeros(len(y_test), dtype=np.float64)

            for run_seed in ensemble_seeds:
                train_ds = (
                    tf.data.Dataset.from_tensor_slices((X_train, y_train))
                    .shuffle(10000, seed=run_seed, reshuffle_each_iteration=True)
                    .batch(batch_size)
                    .prefetch(tf.data.AUTOTUNE)
                )

                model = build_model(u1, a1, d1, u2, a2, d2, run_seed)

                model.fit(
                    train_ds,
                    validation_data=test_ds,
                    epochs=200,
                    class_weight=class_weight,
                    callbacks=make_callbacks(),
                    verbose=0,
                )

                prob_run = model.predict(X_test, batch_size=batch_size, verbose=0).flatten()
                prob_accumulator += prob_run

            # Probabilidad media del ensemble
            probs = (prob_accumulator / n_seeds).astype(np.float32)

            for thr in thresholds:
                preds = (probs >= thr).astype(int)
                tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()

                sensitivity  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                specificity  = tn / (tn + fp) if (tn + fp) > 0 else 0.0
                precision    = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                npv          = tn / (tn + fn) if (tn + fn) > 0 else 0.0
                f1           = f1_score(y_test, preds, zero_division=0)
                acc          = accuracy_score(y_test, preds)
                bal_acc      = balanced_accuracy_score(y_test, preds)

                # FLOOR DURO: descarta directamente modelos degenerados.
                # Un modelo que no supera los mínimos queda en -inf y nunca será elegido.
                if specificity < min_specificity or precision < min_precision:
                    screening_score = float("-inf")
                else:
                    screening_score = (
                        0.70 * sensitivity +
                        0.20 * bal_acc +
                        0.10 * precision
                    )

                row = {
                    "u1": u1, "a1": a1, "d1": d1,
                    "u2": u2, "a2": a2, "d2": d2,
                    "threshold": float(thr),
                    "accuracy": float(acc),
                    "balanced_accuracy": float(bal_acc),
                    "recall_sensitivity": float(sensitivity),
                    "specificity": float(specificity),
                    "precision_ppv": float(precision),
                    "npv": float(npv),
                    "f1_positive": float(f1),
                    "screening_score": float(screening_score),
                    "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
                    "n_seeds_ensemble": n_seeds,
                }
                results.append(row)

                is_better = best is None or row["screening_score"] > best["row"]["screening_score"]
                # Si ambos superan target_recall, desempata con menos FN y mayor especificidad
                if best is not None and row["screening_score"] != float("-inf"):
                    both_high_recall = (
                        row["recall_sensitivity"] >= target_recall and
                        best["row"]["recall_sensitivity"] >= target_recall
                    )
                    if both_high_recall:
                        is_better = (
                            row["FN"] < best["row"]["FN"] or
                            (row["FN"] == best["row"]["FN"] and row["specificity"] > best["row"]["specificity"]) or
                            (row["FN"] == best["row"]["FN"] and row["specificity"] == best["row"]["specificity"] and row["precision_ppv"] > best["row"]["precision_ppv"])
                        )

                if is_better:
                    best = {
                        "row": row.copy(),
                        "model": model,
                        "probs": probs.copy(),
                        "preds": preds.copy(),
                    }

            best_spec = best['row']['specificity'] if best else 0.0
            best_rec  = best['row']['recall_sensitivity'] if best else 0.0
            best_thr  = best['row']['threshold'] if best else 0.0
            best_scr  = best['row']['screening_score'] if best else float('-inf')
            print(
                f"Modelo u1={u1}, a1={a1}, d1={d1}, u2={u2}, a2={a2}, d2={d2} "
                f"(ensemble x{n_seeds}) evaluado. "
                f"Mejor actual: recall={best_rec:.3f}, "
                f"spec={best_spec:.3f}, "
                f"threshold={best_thr:.2f}, "
                f"score={best_scr:.3f}"
            )

    if best is None:
        raise RuntimeError("No se ha podido entrenar/evaluar ningún modelo.")

    df_results = pd.DataFrame(results).sort_values(
        ["screening_score", "recall_sensitivity", "specificity"],
        ascending=False
    ).reset_index(drop=True)
    df_results.to_csv(output_dir / "gridsearch_results_intersection.csv", index=False, encoding="utf-8-sig")

    best_row   = best["row"]
    best_model = best["model"]
    best_probs = best["probs"]
    best_preds = best["preds"]

    print("\n=== MEJOR MODELO SCREENING INTERSECT V2 ===")
    print(pd.Series(best_row).to_string())

    best_model.save(output_dir / "best_model_intersection.keras")

    best_public = {k: v for k, v in best_row.items()}
    best_public["version"] = "V2_MAX_RECALL"
    best_public["use_smote"] = use_smote
    best_public["use_focal_loss"] = use_focal_loss
    best_public["focal_gamma"] = focal_gamma
    best_public["focal_alpha"] = focal_alpha
    best_public["n_seeds_ensemble"] = n_seeds
    best_public["recall_priority_weight"] = recall_priority_weight
    best_public["min_specificity_floor"] = min_specificity
    best_public["min_precision_floor"] = min_precision

    with open(output_dir / "best_config_intersection.json", "w", encoding="utf-8") as f:
        json.dump(best_public, f, indent=2, ensure_ascii=False)

    splits = {
        "train": X_train_df.index.astype(int).tolist(),
        "test": X_test_df.index.astype(int).tolist(),
    }
    with open(output_dir / "splits_indices.json", "w", encoding="utf-8") as f:
        json.dump(splits, f, indent=2)

    X_train_df.to_csv(output_dir / "X_train.csv", index=True, encoding="utf-8-sig")
    X_test_df.to_csv(output_dir / "X_test.csv", index=True, encoding="utf-8-sig")
    y_train_s.to_csv(output_dir / "y_train.csv", index=True, encoding="utf-8-sig")
    y_test_s.to_csv(output_dir / "y_test.csv", index=True, encoding="utf-8-sig")

    df_pred = pd.DataFrame({
        "idx_original": X_test_df.index,
        "y_true_intersection": y_test,
        "y_pred_intersection": best_preds.astype(int),
        "y_prob_intersection": best_probs,
        "threshold_used": best_row["threshold"],
    }).sort_values("idx_original")
    df_pred.to_csv(output_dir / "predictions_with_probs.csv", index=False, encoding="utf-8-sig")

    df_report = report_to_dataframe(y_test, best_preds)
    df_report.to_csv(output_dir / "results_intersection.csv", index=False, encoding="utf-8-sig")

    df_binary = binary_metrics_dataframe(y_test, best_preds, label_name="intersection")
    df_binary.to_csv(output_dir / "binary_metrics_intersection.csv", index=False, encoding="utf-8-sig")

    print("\n=== FINAL REPORT — INTERSECTION V2 ===")
    print(classification_report(y_test, best_preds, digits=3, zero_division=0))

    print("\n=== BINARY METRICS — INTERSECTION V2 ===")
    print(df_binary.to_string(index=False))

    print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_test, best_preds, labels=[0, 1]))

    print("\nSaved files in:", output_dir)
    return df_results, best_public, df_pred, df_report, df_binary


# ============================================================
# 1) Carga y preparación de datos
# ============================================================
feat_df = pd.read_csv("./data/lista_global_vars.csv")
target_df = pd.read_csv("./data/target_col.csv").fillna(0)

print("Dim características:", feat_df.shape)
print("Dim target:", target_df.shape)
print("Valores faltantes:", feat_df.isna().sum().sum() + target_df.isna().sum().sum())
print("Target columns:", target_df.columns.to_list())

df_merged = feat_df.join(target_df, how="inner")
df_merged = (
    df_merged[~((df_merged["GENERO_BIN_2"] == 1) | (df_merged["ORIENTSEX.BN_3"] == 1))]
    .drop(columns=["GENERO_BIN_2", "ORIENTSEX.BN_3"])
    .reset_index(drop=True)
)

df_merged["INTERSECT"] = ((df_merged["VÍCTIMA"] > 0) & (df_merged["PERPETRADOR"] > 0)).astype(int)

print("\n=== CHECK TARGET INTERSECT EN MUESTRA COMPLETA ===")
print(df_merged["INTERSECT"].value_counts().sort_index())
print(df_merged["INTERSECT"].value_counts(normalize=True).sort_index())
print("Total analytical sample after filtering:", len(df_merged))
print("Total INTERSECT:", int(df_merged["INTERSECT"].sum()))

drop_target_cols = [
    "VÍCTIMA", "PERPETRADOR", "VICTIMA_PERPETRADOR",
    "POLIVICTIMIZACION", "POLIPERPETRACION",
    "SOLO.VICTIMA", "SOLO.PERPETRADOR", "NO.VICT_NO.PERP",
    "V.O", "P.SUM.TOTAL", "V.SUM.TOTAL",
]
df_intersection = df_merged.drop(columns=drop_target_cols)
X = df_intersection.drop(columns=["INTERSECT"]).copy()
y = df_intersection["INTERSECT"].astype(int).copy()

pd.set_option("future.no_silent_downcasting", True)

X["PAÍS"] = X["PAÍS"].replace({1: True, 2: False})
X["ETNIA.BN"] = X["ETNIA.BN"].replace({0.0: False, 1.0: True})
X["FUGAS.BN"] = X["FUGAS.BN"].replace({0.0: False, 1.0: True})

X["GENERO.BN0"] = X["GENERO_BIN_0"].replace({0.0: False, 1.0: True})
X["ORIENTSEX.BN0"] = X["ORIENTSEX.BN_1"].replace({0.0: False, 1.0: True})
X["GENERO.BN1"] = X["GENERO_BIN_1"].replace({0.0: False, 1.0: True})
X["ORIENTSEX.BN1"] = X["ORIENTSEX.BN_2"].replace({0.0: False, 1.0: True})
X = X.drop(columns=["GENERO_BIN_0", "GENERO_BIN_1", "ORIENTSEX.BN_1", "ORIENTSEX.BN_2"])

X = X.rename(columns={"CONVIVEN.5": "CONVIVEN_H"})
X["CONVIVEN_H"] = X["CONVIVEN_H"].replace({0.0: False, 1.0: True})
X = X.rename(columns={"CONVIVEN.6": "CONVIVEN_0"})
X["CONVIVEN_0"] = X["CONVIVEN_0"].replace({0.0: False, 1.0: True})

X = X.apply(pd.to_numeric, errors="coerce")
if X.isna().sum().sum() > 0:
    print("WARNING: Hay NaN tras conversión numérica. Se imputan a 0.")
    print(X.isna().sum()[X.isna().sum() > 0])
    X = X.fillna(0)

X.to_csv(OUTPUT_DIR / "df_intersection_feat.csv", index=True, encoding="utf-8-sig")
y.to_csv(OUTPUT_DIR / "df_intersection_target.csv", index=True, encoding="utf-8-sig")

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 2) Split ANTES de scaler/PCA para evitar leakage
# ============================================================
idx_train, idx_test = train_test_split(
    X.index,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

X_train_raw = X.loc[idx_train].copy()
X_test_raw = X.loc[idx_test].copy()
y_train = y.loc[idx_train].copy()
y_test = y.loc[idx_test].copy()

print("\n=== CHECK SPLIT ===")
print("Train size:", len(idx_train), "Test size:", len(idx_test))
print("y_train value_counts:")
print(y_train.value_counts().sort_index())
print("y_test value_counts:")
print(y_test.value_counts().sort_index())


# ============================================================
# 3) PCA train-only, mismo criterio 95% que victim/perp
# ============================================================
X_train_pca, X_test_pca, df_variance, loadings, top_loadings, scaler, pca, means = fit_train_only_pca(
    X_train_raw,
    X_test_raw,
    threshold=PCA_VARIANCE_THRESHOLD,
    output_dir=OUTPUT_DIR,
)

print("\n=== TOP LOADINGS RETAINED PCS ===")
display(top_loadings)


# ============================================================
# 4) Entrenar modelo independiente de overlap/intersection V2
# ============================================================
df_results, best_config, df_pred, df_report, df_binary = train_overlap_nn(
    X_train_pca,
    X_test_pca,
    y_train,
    y_test,
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fast_grid=FAST_GRID,
    seed=SEED,
    recall_priority_weight=1.0,   # 1.0 con SMOTE activo
    min_specificity=0.40,          # floor duro anti-degeneración
    min_precision=0.25,
    use_smote=True,
    use_focal_loss=True,
    focal_gamma=2.0,
    focal_alpha=0.60,              # reducido respecto a v1 para evitar sobre-sesgo positivo
    n_seeds=3,
)

In [ ]:
# CHECK opcional: ejecutar SOLO después de que haya terminado la celda principal.
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

pred_path = Path("./content/intersection_v2/predictions_with_probs.csv")

if pred_path.exists():
    df_pred = pd.read_csv(pred_path)
    print("Saved:", pred_path)
    print(df_pred.head())
    print(df_pred.shape)
    print(df_pred.columns.tolist())

    if "threshold_used" in df_pred.columns:
        print("\nThreshold usado:", df_pred["threshold_used"].iloc[0])

    print("\n=== FINAL REPORT — INTERSECTION BEST CSV V2 ===")
    print(classification_report(df_pred["y_true_intersection"], df_pred["y_pred_intersection"], digits=3, zero_division=0))
    print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(df_pred["y_true_intersection"], df_pred["y_pred_intersection"], labels=[0, 1]))
else:
    print("Todavía no existe:", pred_path)
    print("Ejecuta primero la celda principal que entrena el modelo.")